In [1]:
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

connected to port: 65496


In [2]:
treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-Sequoia"
grew_pattern = "pattern{X[upos=VERB]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/19.investigate_verbs_sequoia/patterns_verb.txt"
analysed_category = "verbs"

In [ ]:
corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    matrix_type="coverage",
    min_occurrences=10,
)

In [ ]:
corpus.feature_matrix.shape

In [ ]:
dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)
clustering = tod.clustering.HierarchicalClustering(corpus)
fig = tod.plotting.cluster_scatter_plot(corpus, dim_red, clustering)

In [ ]:
fig

In [ ]:
fig.write_html("clustering_sequoia_verbs.html")

In [ ]:
simple_fig = tod.plotting.lexunit_scatter_plot(corpus, dim_red)
simple_fig

In [ ]:
simple_fig.write_html("plot_sequoia_verbs.html")

In [ ]:
cluster_assignments = {}
for i in range(1, len(clustering.clusters.keys())+1):
    cluster = clustering.cluster2lexunit(i)
    for lexunit in cluster:
        cluster_assignments[lexunit] = i
len(cluster_assignments.keys())

In [ ]:
patterns_text_file_with_lemmas = "patterns_verbs_with_lemmas.txt"
it_corpus = tod.corpus.IterativeCorpus(
    clusters = cluster_assignments,
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file_with_lemmas,
    matrix_type="coverage",
    min_occurrences=10,
)

In [ ]:
dim_red = tod.dimension_reduction_classic.Tsne_corpus(it_corpus, n_components=2)
clustering = tod.clustering.HierarchicalClustering(it_corpus)
fig = tod.plotting.cluster_scatter_plot(it_corpus, dim_red, clustering)
fig

In [ ]:
fig.write_html("itclustering_sequoia_verbs_with_pos.html")

In [ ]:
it_corpus_no_pos = tod.corpus.IterativeCorpusNoPos(
    clusters = cluster_assignments,
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file_with_lemmas,
    matrix_type="coverage",
    min_occurrences=10,
)

In [ ]:
dim_red = tod.dimension_reduction_classic.Tsne_corpus(it_corpus_no_pos, n_components=2)
clustering = tod.clustering.HierarchicalClustering(it_corpus_no_pos)
fig = tod.plotting.cluster_scatter_plot(it_corpus_no_pos, dim_red, clustering)
fig

In [ ]:
fig.write_html("itclustering_sequoia_verbs_no_pos.html")

In [ ]:
corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    matrix_type="coverage",
    min_occurrences=10,
)

X = corpus.feature_matrix
X.shape

In [ ]:
import pysparcl
k = 47
tidy_k_perm = pysparcl.cluster.permute(X, k=k, nperms=25, nvals=10)

print("Best wbound:", tidy_k_perm['bestw'])

In [ ]:
tidy_k_result = pysparcl.cluster.kmeans(X, k=k, wbounds=tidy_k_perm['bestw'])[0]
tidy_k_weights = tidy_k_result['ws']

In [ ]:
tidy_k_clusters = {i: [] for i in range(k)}
for i in range(len(tidy_k_result['cs'])):
    cluster_id = tidy_k_result['cs'][i]
    tidy_k_clusters[cluster_id].append(corpus.idx2lexunit(i))

In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)
# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_tsne[:, 0],
    'PCA2': X_tsne[:, 1],
    'Cluster': tidy_k_result['cs'],
    'Word': [corpus.idx2lexunit(i) for i in range(len(tidy_k_result['cs']))]
})
fig = go.Figure()
for cluster in range(k):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Word Clusters',
    xaxis_title='tsne1',
    yaxis_title='tsne2',
    width=1600,  # Set the width of the figure
    height=800,  # Set the height of the figure

    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * k},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(k)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, k + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

In [ ]:
import numpy as np
tidy_k_important_features = np.argsort(-tidy_k_weights) 
for i in tidy_k_important_features[:10]:  
    print(f"{corpus.idx2feature(i)}: weight {tidy_k_weights[i]:.3f}")

The problem is that the silhouette score determined that 47 is the ideal number of clusters. So we can't really launch grex for each of the 47 clusters because life is too short. 

What we can do is divide the verbs in max 2 clusters and then see what happens. 

#### Division of verbs in 2 classes

In [3]:
corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    matrix_type="coverage",
    min_occurrences=10,
)

dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)
clustering = tod.clustering.KMeans(corpus, k=2)
fig = tod.plotting.cluster_scatter_plot(corpus, dim_red, clustering)
fig

Number of matches after filtering: 3504


/opt/homebrew/lib/python3.11/site-packages/kaleido/__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [4]:
X = corpus.feature_matrix
X.shape

(134, 264)

In [5]:
import pysparcl
k = 2
tidy_k_perm = pysparcl.cluster.permute(X, k=k, nperms=25, nvals=10)

print("Best wbound:", tidy_k_perm['bestw'])

tidy_k_result = pysparcl.cluster.kmeans(X, k=k, wbounds=tidy_k_perm['bestw'])[0]
tidy_k_weights = tidy_k_result['ws']

tidy_k_clusters = {i: [] for i in range(k)}
for i in range(len(tidy_k_result['cs'])):
    cluster_id = tidy_k_result['cs'][i]
    tidy_k_clusters[cluster_id].append(corpus.idx2lexunit(i))

from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)
# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_tsne[:, 0],
    'PCA2': X_tsne[:, 1],
    'Cluster': tidy_k_result['cs'],
    'Word': [corpus.idx2lexunit(i) for i in range(len(tidy_k_result['cs']))]
})
fig = go.Figure()
for cluster in range(k):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Word Clusters',
    xaxis_title='tsne1',
    yaxis_title='tsne2',
    # width=1600,  # Set the width of the figure
    # height=800,  # Set the height of the figure

    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * k},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(k)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, k + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

Best wbound: 4.81325193712318


In [6]:
import numpy as np
tidy_k_important_features = np.argsort(-tidy_k_weights) 
for i in tidy_k_important_features[:10]:  
    print(f"{corpus.idx2feature(i)}: weight {tidy_k_weights[i]:.3f}")

node:X:own:Voice=Pass: weight 0.575
node:X:own:VerbForm=Part: weight 0.475
node:X:own:Tense=Past: weight 0.414
node:X:child:rel_shallow=nsubj: weight 0.210
node:X:own:VerbForm=Fin: weight 0.195
node:X:own:rel_shallow=acl: weight 0.170
node:X:own:Mood=Ind: weight 0.159
node:X:parent:upos=NOUN: weight 0.136
node:X:next:upos=ADP: weight 0.132
node:X:own:Person=3: weight 0.123


In [9]:
cluster_assignments = {}
for orig_c, l in tidy_k_clusters.items():
    for lexunit in l:
        cluster_assignments[lexunit] = orig_c

In [10]:
import csv

  
with open("sequoia_verb_cluster_assignments.csv", "w") as f:

	writer = csv.writer(f)

	for lexunit, cluster in cluster_assignments.items():

		writer.writerow([lexunit, cluster])